# Function 4: Black-box optimisation

This notebook documents 13 rounds of model fitting, candidate selection and evaluated outputs for Function 4. Alternative approaches are retained where they informed the weekly decisions.

**Main result:** A Matérn GP found the best observed output of 0.625128 in Week 5.

**Usage:** Run the setup and then the weekly sections in order. The initial datasets are in this folder; the full query history is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv). Some early searches generate large candidate grids and may require substantial memory.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.optimize import NonlinearConstraint, differential_evolution
from scipy.spatial import cKDTree
from scipy.stats import truncnorm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, RBF, WhiteKernel


def resolve_initial_path(filename: str) -> Path:
    """Find the initial dataset for Function 4."""
    for folder in (Path.cwd(), Path.cwd() / "function_4"):
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"{filename} not found. Place it in the function_4/ folder."
    )

## Reusable routines

The helper functions below handle standard GP fitting, UCB scoring and recording evaluated queries. The model settings and experiments remain visible within each week. Global/local candidate generation is shared across the early rounds.

In [ ]:
def fit_gp_model(X, y, **settings):
    """Fit a GP using the explicit kernel and settings chosen for a given week."""
    gp = GaussianProcessRegressor(**settings)
    gp.fit(X, y)
    return gp


def ucb_score(mean, std, beta):
    """Upper Confidence Bound: predictive mean plus weighted uncertainty."""
    return mean + beta * std


def add_evaluation(X, y, x_observed, y_observed):
    """Add one completed black-box evaluation to the training history."""
    x_row = np.asarray(x_observed, dtype=float).reshape(1, -1)
    y_value = np.asarray(y_observed, dtype=float).reshape(1)
    if x_row.shape[1] != X.shape[1]:
        raise ValueError("The evaluated input has the wrong number of dimensions.")
    return np.vstack([X, x_row]), np.concatenate([y, y_value])

def sample_hybrid_candidates(best_x, n_global, n_local, seed, *,
                             local_scale=0.08, local_method="truncated"):
    """Combine uniform global samples with local samples near the current best.

    The local method is either "truncated" (truncated normal, inside [0, 1])
    or "clipped" (normal samples clipped to [0, 1]). Random number generation
    follows the same sequence for a given seed as the weekly experiments.
    """
    best_x = np.asarray(best_x, dtype=float).reshape(-1)
    n_features = best_x.size
    rng = np.random.default_rng(seed)

    X_global = rng.uniform(0, 1, size=(n_global, n_features))
    if local_method == "clipped":
        X_local = rng.normal(
            loc=best_x, scale=local_scale, size=(n_local, n_features)
        )
        X_local = np.clip(X_local, 0, 1)
    elif local_method == "truncated":
        X_local = np.empty((n_local, n_features))
        for j in range(n_features):
            a = (0 - best_x[j]) / local_scale
            b = (1 - best_x[j]) / local_scale
            X_local[:, j] = truncnorm.rvs(
                a, b, loc=best_x[j], scale=local_scale,
                size=n_local, random_state=rng
            )
    else:
        raise ValueError("local_method must be 'clipped' or 'truncated'.")

    return np.round(np.vstack([X_global, X_local]), 6)


## Starting data

In [2]:
X = np.load(resolve_initial_path("initial_inputs.npy"))
y = np.load(resolve_initial_path("initial_outputs.npy"))

## Week 1

In [6]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

# UCB exploration parameter
beta = 1.96
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds="fixed"
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True
)
# Generate candidate inputs
# Define the search range for each input variable

x1_values = np.linspace(0, 1, 70)
x2_values = np.linspace(0, 1, 70)
x3_values = np.linspace(0, 1, 70)
x4_values = np.linspace(0, 1, 70)

# Generate every combination of x1 and x2
X1_grid, X2_grid, X3_grid, X4_grid = np.meshgrid(
    x1_values,
    x2_values,
    x3_values,
    x4_values
)

X_grid = np.column_stack([
    X1_grid.ravel(),
    X2_grid.ravel(),
    X3_grid.ravel(),
    X4_grid.ravel()
])
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_grid,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)
# Select the next query
best_index = np.argmax(UCB)

next_X = X_grid[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.49275362 0.44927536 0.44927536 0.20289855]

Predicted mean:
-8.113146748772825

Predicted standard deviation:
5.518310949908169

UCB:
2.7027427130471846


### Evaluated query

**Input:** `(0.492754, 0.449275, 0.449275, 0.202899)`  
**Output:** `-4.222249616167833`

In [ ]:
X_week1 = np.array([0.492754, 0.449275, 0.449275, 0.202899])
y_week1 = np.array([-4.222249616167833])
X_new, y_new = add_evaluation(X, y, X_week1, y_week1)

## Week 2

In [10]:
X = X_new
y = y_new

### Increase β to favour exploration

In [11]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 3

### Allow the kernel length scale to vary

In [12]:
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.518)


### Use random candidates instead of a grid

In [13]:
# Generate candidate inputs
n_candidates = 1000000
n_features = 4

np.random.seed(42)

X_candidates = np.random.uniform(
    low=0,
    high=1,
    size=(n_candidates, n_features)
)

X_candidates = np.round(X_candidates, 6)

In [14]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [15]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.42217  0.440988 0.315755 0.458067]

Predicted mean:
-1.174289124091704

Predicted standard deviation:
0.9439772064888

UCB:
1.657642495374696


### Evaluated query

**Input:** `(0.422170, 0.440988, 0.315755, 0.458067)`  
**Output:** `-0.8407000722334064`

In [ ]:
X_week2 = np.array([0.42217, 0.440988, 0.315755, 0.458067])
y_week2 = np.array([-0.8407000722334064])
X_new, y_new = add_evaluation(X, y, X_week2, y_week2)

## Week 3

In [19]:
X = X_new
y = y_new

### Option 1: Isotropic GP with global/local sampling · selected

In [20]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 2.5

In [21]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.542)


In [22]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="clipped",
)

In [23]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [24]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.407937 0.428034 0.422853 0.446386]

Predicted mean:
-0.9245673554894598

Predicted standard deviation:
0.3821204144145248

UCB:
0.030733680546852327


### Option 2: Anisotropic GP with global/local sampling

In [25]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1]

beta = 2.5

In [26]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.01, 2.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[0.568, 0.478, 0.618, 0.533])


In [27]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="clipped",
)

In [28]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [29]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.474954 0.500656 0.252054 0.41278 ]

Predicted mean:
-1.2292002739951862

Predicted standard deviation:
0.4769086903349864

UCB:
-0.036928548157720176


### Evaluated query

**Input:** `(0.407937, 0.428034, 0.422853, 0.446386)`  
**Output:** `0.2601671984894094`

In [ ]:
X_week3 = np.array([0.407937, 0.428034, 0.422853, 0.446386])
y_week3 = np.array([0.2601671984894094])
X_new, y_new = add_evaluation(X, y, X_week3, y_week3)

## Week 4

In [33]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 0.5 · selected

In [34]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 0.5

In [35]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.501)


In [36]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [37]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [38]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.408036 0.450886 0.429189 0.451204]

Predicted mean:
0.268504852654198

Predicted standard deviation:
0.09064842790775267

UCB:
0.3138290666080743


### Option 2: Anisotropic GP with β = 0.5 · alternative

In [39]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1]

beta = 0.5

In [40]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.01, 2.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[0.589, 0.493, 0.482, 0.491])


In [41]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [42]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [43]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.393582 0.452112 0.43367  0.452499]

Predicted mean:
0.2910662924717844

Predicted standard deviation:
0.10832852883852125

UCB:
0.34523055689104504


### Evaluated query

**Input:** `(0.408036, 0.450886, 0.429189, 0.451204)`  
**Output:** `-0.3006069539784799`

In [ ]:
X_week4 = np.array([0.408036, 0.450886, 0.429189, 0.451204])
y_week4 = np.array([-0.3006069539784799])
X_new, y_new = add_evaluation(X, y, X_week4, y_week4)

## Week 5

In [47]:
X = X_new
y = y_new

In [48]:
# Week 5 GP parameters
beta = 1.0

kernel = Matern(
    length_scale=0.3,
    length_scale_bounds=(0.02, 2.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

## Global GP-UCB acquisition
#def negative_ucb(x):
#    mu, std = model.predict(
#        np.asarray(x).reshape(1, -1),
#        return_std=True
#    )
#    return -(mu[0] + beta * std[0])

#result = differential_evolution(
#    negative_ucb,
#    bounds=[(0, 1)] * 4,
#    seed=46,
#    popsize=20,
#    maxiter=150,
#    tol=1e-8,
#    polish=True
#)

#next_X = np.round(result.x, 6)

#mu, std = model.predict(
#    next_X.reshape(1, -1),
#    return_std=True
#)

#print("Optimised kernel:", model.kernel_)
#print("Week 5 candidate:", next_X)
#print("Predicted mean:", mu[0])
#print("Predicted std:", std[0])
#print("UCB:", mu[0] + beta * std[0])

,kernel,"Matern(length...e=0.3, nu=1.5)"
,alpha,1e-10
,optimizer,'fmin_l_bfgs_b'
,n_restarts_optimizer,10
,normalize_y,True
,copy_X_train,True
,n_targets,None
,random_state,42
,kernel__length_scale,0.3
,kernel__length_scale_bounds,"(0.02, ...)"
,kernel__nu,1.5


In [49]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=47,
    local_scale=0.08,
    local_method="truncated",
)

In [50]:
# Predict at existing candidate points
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)

# Calculate GP-UCB
beta = 1.0
UCB = ucb_score(post_mean, post_std, beta)

# Select the highest-UCB candidate
best_index = np.argmax(UCB)
next_X = X_candidates[best_index]


print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.428134 0.353942 0.402952 0.414956]

Predicted mean:
0.6372948926073647

Predicted standard deviation:
0.6138691607652785

UCB:
1.2511640533726434


### Evaluated query

**Input:** `(0.428134, 0.353942, 0.402952, 0.414956)`  
**Output:** `0.6251275893252033`

In [ ]:
X_week5 = np.array([0.428134, 0.353942, 0.402952, 0.414956])
y_week5 = np.array([0.6251275893252033])
X_new, y_new = add_evaluation(X, y, X_week5, y_week5)

## Week 6

In [54]:
X = X_new
y = y_new

### Isotropic Matérn 3/2 kernel with β = 1 · selected

In [55]:
# Week 6 GP parameters
beta = 1.0

kernel = Matern(
    length_scale=0.3,
    length_scale_bounds=(0.02, 2.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

,kernel,"Matern(length...e=0.3, nu=1.5)"
,alpha,1e-10
,optimizer,'fmin_l_bfgs_b'
,n_restarts_optimizer,10
,normalize_y,True
,copy_X_train,True
,n_targets,None
,random_state,42
,kernel__length_scale,0.3
,kernel__length_scale_bounds,"(0.02, ...)"
,kernel__nu,1.5


In [56]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 4
n_global = 700000
n_local = 300000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=48,
    local_scale=0.08,
    local_method="truncated",
)

In [57]:
# Predict at existing candidate points
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)

# Calculate GP-UCB
beta = 1.0
UCB = ucb_score(post_mean, post_std, beta)

# Select the highest-UCB candidate
best_index = np.argmax(UCB)
next_X = X_candidates[best_index]


print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.382209 0.375536 0.407365 0.416335]

Predicted mean:
0.6090414466385266

Predicted standard deviation:
0.40631864876109125

UCB:
1.0153600953996178


### Evaluated query

**Input:** `(0.382209, 0.375536, 0.407365, 0.416335)`  
**Output:** `0.46377468126707244`

In [ ]:
X_week6 = np.array([0.382209, 0.375536, 0.407365, 0.416335])
y_week6 = np.array([0.46377468126707244])
X_new, y_new = add_evaluation(X, y, X_week6, y_week6)

## Week 7

In [61]:
# All observations through Week 6
X = X_new
y = y_new

assert X.shape == (36, 4)
assert y.shape == (36,)

# Selected GP hyperparameters
kernel = Matern(
    length_scale=0.5,
    length_scale_bounds=(0.02, 4.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)
print("Optimised kernel:", model.kernel_)

# GP-UCB acquisition parameters
beta = 1.0
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(point):
    point = np.asarray(point)

    mu, std = model.predict(
        point.reshape(1, -1),
        return_std=True
    )

    distance = tree.query(point)[0]

    # Penalise candidate points too close to observations
    penalty = 500 * max(0, min_distance - distance)

    return -(mu[0] + beta * std[0]) + penalty

# Search the acquisition function
solutions = []

for seed in [42, 48]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 4,
        seed=seed,
        popsize=18,
        maxiter=150,
        tol=1e-7,
        polish=True
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError("No feasible candidate found.")

best_result = min(solutions, key=lambda r: r.fun)
next_X = np.round(best_result.x, 6)

mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

print("Week 7 recommended query:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print("Nearest observation distance:", tree.query(next_X)[0])

Optimised kernel: Matern(length_scale=0.916, nu=1.5)
Week 7 recommended query: [0.42709  0.367306 0.412696 0.462298]
Predicted mean: 0.6246230891277129
Predicted std: 0.36314545596309666
UCB: 0.9877685450908096
Nearest observation distance: 0.050158717407844486


### Evaluated query

**Input:** `(0.427090, 0.367306, 0.412696, 0.462298)`  
**Output:** `-0.24008611800206792`

In [ ]:
X_week7 = np.array([0.42709 , 0.367306, 0.412696, 0.462298])
y_week7 = np.array([-0.24008611800206792])
X_new, y_new = add_evaluation(X, y, X_week7, y_week7)

## Week 8

In [65]:
# All observations through Week 7
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (37, 4)
assert y.shape == (37,)

# Matérn 3/2 with fitted signal amplitude
# and noise regularisation
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 200.0)
    )
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 5.0),
        nu=1.5
    )
    + WhiteKernel(
        noise_level=1e-4,
        noise_level_bounds=(1e-6, 0.3)
    )
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)
print("Optimised kernel:", model.kernel_)

# GP-UCB settings
beta = 1.0
min_distance = 0.06
tree = cKDTree(X)

# Fitted WhiteKernel variance on original scale
noise_variance = (
    model.kernel_.k2.noise_level * np.var(y)
)

def latent_predict(points):
    mu, total_std = model.predict(
        points,
        return_std=True
    )

    latent_std = np.sqrt(
        np.maximum(
            total_std**2 - noise_variance,
            0.0
        )
    )

    return mu, latent_std

def negative_ucb(points):
    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = latent_predict(candidates)
    distances = tree.query(candidates)[0]

    # Penalise locations too close to observations
    penalty = 100 * np.maximum(
        0, min_distance - distances
    )

    scores = -(ucb_score(mu, std, beta)) + penalty

    return scores if points.ndim == 2 else scores[0]

# Optimise with multiple seeds
solutions = []

for seed in [42, 48, 59]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 4,
        seed=seed,
        popsize=18,
        maxiter=180,
        tol=1e-8,
        vectorized=True,
        updating="deferred",
        polish=False
    )

    if tree.query(result.x)[0] >= min_distance - 1e-5:
        solutions.append(result)

if not solutions:
    raise RuntimeError("No feasible candidate found.")

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(best_result.x, 6)

mu, std = latent_predict(next_X.reshape(1, -1))

print("Week 8 query:", next_X)
print("Predicted mean:", mu[0])
print("Latent predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print("Nearest distance:", tree.query(next_X)[0])

Optimised kernel: 5.09**2 * Matern(length_scale=3.71, nu=1.5) + WhiteKernel(noise_level=0.000584)
Week 8 query: [0.416788 0.39469  0.392609 0.373678]
Predicted mean: 0.43719226503651676
Latent predicted std: 0.30059391305833166
UCB: 0.7377861780948485
Nearest distance: 0.059999517848062726


### Evaluated query

**Input:** `(0.416788, 0.394690, 0.392609, 0.373678)`  
**Output:** `0.33191258998986983`

In [ ]:
X_week8 = np.array([0.416788, 0.39469 , 0.392609, 0.373678])
y_week8 = np.array([0.33191258998986983])
X_new, y_new = add_evaluation(X, y, X_week8, y_week8)

## Week 9

In [69]:
# Observations through Week 8
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (38, 4)
assert y.shape == (38,)
# Fit selected GP model
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 200.0)
    )
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 5.0),
        nu=1.5
    )
    + WhiteKernel(
        noise_level=1e-4,
        noise_level_bounds=(1e-6, 0.3)
    )
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Optimised kernel:", model.kernel_)
# Latent-function predictions
noise_variance = (
    model.kernel_.k2.noise_level * np.var(y)
)

def latent_predict(points):
    mu, total_std = model.predict(
        points,
        return_std=True
    )

    latent_std = np.sqrt(
        np.maximum(
            total_std**2 - noise_variance,
            0.0
        )
    )

    return mu, latent_std
# Acquisition parameters
beta = 1.5
min_distance = 0.06
tree = cKDTree(X)

def negative_ucb(points):

    points = np.asarray(points)

    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = latent_predict(candidates)
    distances = tree.query(candidates)[0]

    ucb = ucb_score(mu, std, beta)

    penalty = 500 * np.maximum(
        0,
        min_distance - distances
    )

    scores = -ucb + penalty

    return (
        scores if points.ndim == 2
        else scores[0]
    )
# Optimise the acquisition function
solutions = []

for seed in [42, 59]:

    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 4,
        seed=seed,
        popsize=13,
        maxiter=140,
        tol=1e-7,
        vectorized=True,
        updating="deferred",
        polish=False
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible candidate found."
    )

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(best_result.x, 6)
# Report Week 9 query
mu, std = latent_predict(
    next_X.reshape(1, -1)
)

print("Week 9 query:", next_X)
print("Predicted mean:", mu[0])
print("Latent predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print(
    "Nearest observation distance:",
    tree.query(next_X)[0]
)
print("Current best output:", y.max())

Optimised kernel: 5.25**2 * Matern(length_scale=3.89, nu=1.5) + WhiteKernel(noise_level=0.000528)
Week 9 query: [0.411485 0.371724 0.447096 0.382424]
Predicted mean: 0.26348959814191986
Latent predicted std: 0.28900115456944936
UCB: 0.6969913299960939
Nearest observation distance: 0.06000103167446374
Current best output: 0.6251275893252033


### Evaluated query

**Input:** `(0.411485, 0.371724, 0.447096, 0.382424)`  
**Output:** `0.03065042722702005`

In [ ]:
X_week9 = np.array([0.411485, 0.371724, 0.447096, 0.382424])
y_week9 = np.array([0.03065042722702005])
X_new, y_new = add_evaluation(X, y, X_week9, y_week9)

## Week 10

In [73]:
# All observations through Week 9
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (39, 4)
assert y.shape == (39,)
# Selected GP model
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 200.0)
    )
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 5.0),
        nu=1.5
    )
    + WhiteKernel(
        noise_level=1e-4,
        noise_level_bounds=(1e-6, 0.3)
    )
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Optimised kernel:", model.kernel_)
# Latent-function predictions
noise_variance = (
    model.kernel_.k2.noise_level * np.var(y)
)

def latent_predict(points):
    mean, total_std = model.predict(
        points,
        return_std=True
    )

    latent_std = np.sqrt(
        np.maximum(
            total_std**2 - noise_variance,
            0.0
        )
    )

    return mean, latent_std
# Acquisition settings
beta = 1.5
min_distance = 0.06
tree = cKDTree(X)

def negative_ucb(x):
    x = np.asarray(x).reshape(1, -1)

    mu, std = latent_predict(x)

    return -float(mu[0] + beta * std[0])

# Explicit minimum-distance constraint
distance_constraint = NonlinearConstraint(
    lambda x: tree.query(x)[0],
    min_distance + 1e-6,
    np.inf
)
# Optimise GP-UCB
solutions = []

for seed in [42, 59, 111]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 4,
        constraints=distance_constraint,
        seed=seed,
        popsize=15,
        maxiter=180,
        tol=1e-7,
        polish=True
    )

    # Check actual constraint satisfaction,
    # including if maxiter was reached
    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible candidate found."
    )

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(best_result.x, 6)
# Report Week 10 query
mu, std = latent_predict(
    next_X.reshape(1, -1)
)

print("Week 10 query:", next_X)
print("Predicted mean:", mu[0])
print("Latent std:", std[0])
print("GP-UCB:", mu[0] + beta * std[0])

print(
    "Nearest observation distance:",
    tree.query(next_X)[0]
)

print("Current best:", y.max())

Optimised kernel: 5.32**2 * Matern(length_scale=3.99, nu=1.5) + WhiteKernel(noise_level=0.000518)
Week 10 query: [0.395761 0.355935 0.357173 0.393685]
Predicted mean: 0.2467039006091838
Latent std: 0.2841315189993814
GP-UCB: 0.6729011791082559
Nearest observation distance: 0.0600006241550869
Current best: 0.6251275893252033


### Evaluated query

**Input:** `(0.395761, 0.355935, 0.357173, 0.393685)`  
**Output:** `0.5102550445227156`

In [ ]:
X_week10 = np.array([0.395761, 0.355935, 0.357173, 0.393685])
y_week10 = np.array([0.5102550445227156])
X_new, y_new = add_evaluation(X, y, X_week10, y_week10)

## Week 11

In [76]:
# Observations through Week 10
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (40, 4)
assert y.shape == (40,)

best_index = np.argmax(y)
best_X = X[best_index]
best_y = y[best_index]

print("Best observed input:", best_X)
print("Best observed output:", best_y)
# Fit selected GP
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 200.0)
    )
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 5.0),
        nu=1.5
    )
    + WhiteKernel(
        noise_level=1e-4,
        noise_level_bounds=(1e-6, 0.3)
    )
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-8,
    normalize_y=True,
    n_restarts_optimizer=3,
    random_state=42
)

print("Optimised kernel:", model.kernel_)
# Local trust region
radius = 0.12
min_distance = 0.02

bounds = [
    (
        max(0, coordinate - radius),
        min(1, coordinate + radius)
    )
    for coordinate in best_X
]

tree = cKDTree(X)
# Acquisition: predicted mean
def negative_mean(point):

    point = np.asarray(point)

    mu = model.predict(
        point.reshape(1, -1)
    )[0]

    distance = tree.query(point)[0]

    penalty = 100 * max(
        0,
        min_distance - distance
    )

    return -mu + penalty
# Optimise the acquisition function
result = differential_evolution(
    negative_mean,
    bounds=bounds,
    seed=42,
    popsize=12,
    maxiter=130,
    tol=1e-5,
    polish=True
)

next_X = np.round(result.x, 6)
# Calculate latent GP uncertainty
mu, total_std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

noise_variance = (
    model.kernel_.k2.noise_level
    * model._y_train_std**2
)

latent_std = np.sqrt(
    max(total_std[0]**2 - noise_variance, 0)
)

beta = 0.5
ucb = mu[0] + beta * latent_std
# Report recommendation
print("\nWeek 11 proposed input:", next_X)
print("Predicted mean:", mu[0])
print("Latent predicted std:", latent_std)
print("GP-UCB:", ucb)
print(
    "Nearest observation distance:",
    tree.query(next_X)[0]
)
print("Current best output:", best_y)

Best observed input: [0.428134 0.353942 0.402952 0.414956]
Best observed output: 0.6251275893252033
Optimised kernel: 5.33**2 * Matern(length_scale=4.03, nu=1.5) + WhiteKernel(noise_level=0.000421)

Week 11 proposed input: [0.40284  0.3742   0.386734 0.402159]
Predicted mean: 0.558673509187722
Latent predicted std: 0.11353524100866692
GP-UCB: 0.6154411296920554
Nearest observation distance: 0.032465677168357306
Current best output: 0.6251275893252033


### Evaluated query

**Input:** `(0.402840, 0.374200, 0.386734, 0.402159)`  
**Output:** `0.22325829985369383`

In [ ]:
X_week11 = np.array([0.40284 , 0.3742  , 0.386734, 0.402159])
y_week11 = np.array([0.22325829985369383])
X_new, y_new = add_evaluation(X, y, X_week11, y_week11)

## Week 12

In [79]:
# Observations through Week 11
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (41, 4)
assert y.shape == (41,)

best_idx = np.argmax(y)
best_X = X[best_idx]
best_y = y[best_idx]

print("Current best:", best_X, best_y)
# Fit regularised GP
base_kernel = (
    ConstantKernel(1.0, (0.03, 200.0))
    * Matern(
        length_scale=0.6,
        length_scale_bounds=(0.02, 8.0),
        nu=1.5
    )
)

gp_regularised = fit_gp_model(
    X, y,
    kernel=base_kernel + WhiteKernel(
        noise_level=0.001,
        noise_level_bounds=(1e-6, 0.3)
    ),
    alpha=1e-8,
    normalize_y=True,
    n_restarts_optimizer=2,
    random_state=42
)
# Fit interpolating GP
gp_interpolating = fit_gp_model(
    X, y,
    kernel=base_kernel,
    alpha=1e-7,
    normalize_y=True,
    n_restarts_optimizer=2,
    random_state=42
)
# Controlled change to fourth variable
next_X = best_X.copy()

next_X[3] -= 0.015

next_X = np.round(
    np.clip(next_X, 0, 1),
    6
)

tree = cKDTree(X)
nearest_distance = tree.query(next_X)[0]

assert nearest_distance >= 0.014
# Compare model predictions
print("\nWeek 12 proposed input:")
print(" ".join(f"{v:.6f}" for v in next_X))

for name, gp in [
    ("Regularised GP", gp_regularised),
    ("Interpolating GP", gp_interpolating)
]:
    mu, std = gp.predict(
        next_X.reshape(1, -1),
        return_std=True
    )

    print("\n", name)
    print("Kernel:", gp.kernel_)
    print("Predicted mean:", mu[0])
    print("Predicted std:", std[0])

print("\nCurrent best output:", best_y)
print("Nearest observation distance:", nearest_distance)

Current best: [0.428134 0.353942 0.402952 0.414956] 0.6251275893252033

Week 12 proposed input:
0.428134 0.353942 0.402952 0.399956

 Regularised GP
Kernel: 5.49**2 * Matern(length_scale=4.18, nu=1.5) + WhiteKernel(noise_level=0.00054)
Predicted mean: 0.3603374091883609
Predicted std: 0.2697015970460934

 Interpolating GP
Kernel: 1.45**2 * Matern(length_scale=0.972, nu=1.5)
Predicted mean: 0.6507276698468463
Predicted std: 0.0971860525759044

Current best output: 0.6251275893252033
Nearest observation distance: 0.015000000000000013


### Evaluated query

**Input:** `(0.428134, 0.353942, 0.402952, 0.399956)`  
**Output:** `0.54805267638945`

In [ ]:
X_week12 = np.array([0.428134, 0.353942, 0.402952, 0.399956])
y_week12 = np.array([0.54805267638945])
X_new, y_new = add_evaluation(X, y, X_week12, y_week12)

## Week 13

In [83]:
# Observations through Week 12
X_all = np.asarray(X_new, dtype=float)
y_all = np.asarray(y_new, dtype=float).ravel()

assert X_all.shape == (42, 4)
assert y_all.shape == (42,)

best_idx = np.argmax(y_all)

best_X = X_all[best_idx]
best_y = y_all[best_idx]

print("Best observed input:", best_X)
print("Best observed output:", best_y)
# Fit complementary GP models
kernels = {
    "Matern 3/2": Matern(
        length_scale=0.4,
        length_scale_bounds=(0.005, 10),
        nu=1.5
    ),
    "Matern 5/2": Matern(
        length_scale=0.4,
        length_scale_bounds=(0.005, 10),
        nu=2.5
    ),
    "Anisotropic Matern 3/2": Matern(
        length_scale=[0.3] * 4,
        length_scale_bounds=(0.005, 10),
        nu=1.5
    ),
    "Anisotropic Matern 5/2": Matern(
        length_scale=[0.3] * 4,
        length_scale_bounds=(0.005, 10),
        nu=2.5
    ),
    "Regularised Matern 3/2": (
        ConstantKernel(1.0, (0.03, 200))
        * Matern(
            length_scale=0.6,
            length_scale_bounds=(0.02, 8),
            nu=1.5
        )
        + WhiteKernel(
            noise_level=0.001,
            noise_level_bounds=(1e-6, 0.3)
        )
    )
}

models = {}

for name, kernel in kernels.items():

    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-7,
        normalize_y=True,
        n_restarts_optimizer=1,
        random_state=42
    )

    model.fit(X_all, y_all)

    models[name] = model
# Candidate: decrease x2 by 0.010
next_X = best_X.copy()
next_X[1] -= 0.010

next_X = np.round(
    np.clip(next_X, 0, 1),
    6
)
# Compare predictions
print("\nPredictions:")

for name, model in models.items():

    mu, std = model.predict(
        next_X.reshape(1, -1),
        return_std=True
    )

    print(
        f"{name}: "
        f"mean={mu[0]:.6f}, "
        f"std={std[0]:.6f}"
    )
# Distances and final coordinates
tree = cKDTree(X_all)

print("\nNearest observation distance:",
      tree.query(next_X)[0])

print("Distance from current best:",
      np.linalg.norm(next_X - best_X))

print("\nFINAL WEEK 13 SUBMISSION:")

print(
    " ".join(f"{v:.6f}" for v in next_X)
)

Best observed input: [0.428134 0.353942 0.402952 0.414956]
Best observed output: 0.6251275893252033

Predictions:
Matern 3/2: mean=0.642405, std=0.090661
Matern 5/2: mean=0.694134, std=0.104913
Anisotropic Matern 3/2: mean=0.747444, std=0.129820
Anisotropic Matern 5/2: mean=1.080650, std=0.132931
Regularised Matern 3/2: mean=0.355156, std=0.252408

Nearest observation distance: 0.009999999999999953
Distance from current best: 0.009999999999999953

FINAL WEEK 13 SUBMISSION:
0.428134 0.343942 0.402952 0.414956


### Week 13 evaluated result

**Input:** `(0.428134, 0.343942, 0.402952, 0.414956)`  
**Output:** `0.5218830722339125`

The query-selection calculations above use results available through Week 12. The full record is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv).